In [1]:
# Import library here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df_train_clean = pd.read_csv("../../data/processed/01.3/accidents_advance_clean.csv")

df_train_clean.head(10)

,Severity,Start_Time,Start_Lat,Start_Lng,Distance(mi),Description,Street,City,County,State,...,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,Duration(min),Start_Date,Hour,Month,Weather_Group
0,3,2016-11-08 08:50:37,29.718655,-95.321053,0.009950,Accident on I-45 Northbound at I-45 Exits 43A ...,I-45 N,Houston,Harris,TX,...,False,Day,Day,Day,Day,3.405079,2016-11-08,8,11,Cloudy
1,2,2021-07-26 07:04:56,38.927551,-121.080093,0.000000,Right hand shoulder blocked due to accident on...,Taylor Ln,Auburn,Placer,CA,...,False,Day,Day,Day,Day,3.862833,2021-07-26,7,7,Clear
2,2,2018-10-24 08:23:38,34.776241,-86.672829,0.000000,Lane blocked due to accident on AL-255 Researc...,Highway 255,Huntsville,Madison,AL,...,False,Day,Day,Day,Day,3.422633,2018-10-24,8,10,Clear
3,3,2019-07-11 10:42:48,42.384880,-83.149467,0.470004,Entry ramp to I-96 Westbound from Davison West...,Edward J Jeffries Fwy,Detroit,Wayne,MI,...,False,Day,Day,Day,Day,4.784571,2019-07-11,10,7,Clear
4,2,2020-11-06 01:29:00,33.776575,-117.837134,0.387980,1023: NB 55 JNO 22. SV REAR ENDED VV AND WAS L...,Garden Grove Fwy,Orange,Orange,CA,...,False,Night,Night,Night,Night,4.952535,2020-11-06,1,11,Clear
5,2,2021-07-02 15:22:00,36.989497,-121.935657,0.033435,Incident on PARK AVE WB near SOQUEL DR Expect ...,Park Ave N,Soquel,Santa Cruz,CA,...,False,Day,Day,Day,Day,5.758376,2021-07-02,15,7,Cloudy
6,2,2019-08-09 08:44:18,30.419380,-91.096611,0.000000,Accident on LA-73 Jefferson Hwy at LA-3064 Ess...,Essen Ln,Baton Rouge,East Baton Rouge,LA,...,False,Day,Day,Day,Day,4.665638,2019-08-09,8,8,Cloudy
7,2,2021-12-10 07:02:30,33.194519,-111.901301,1.015955,Stationary traffic on I-10 E - Pearl Harbor Me...,I-10 E,Maricopa,Pinal,AZ,...,False,Night,Day,Day,Day,4.248495,2021-12-10,7,12,Clear
8,3,2021-04-26 19:57:38,29.384928,-98.512321,0.000000,Three lanes blocked due to accident on I-35 So...,I-35 N,San Antonio,Bexar,TX,...,False,Day,Day,Day,Day,5.332880,2021-04-26,19,4,Clear
9,2,2017-04-18 19:05:08,26.589575,-81.703766,0.000000,Accident on 25th St at Curtis Ave.,25th St SW,Lehigh Acres,Lee,FL,...,False,Day,Day,Day,Day,3.751463,2017-04-18,19,4,Cloudy


In [3]:
from sklearn.preprocessing import StandardScaler

df_train = pd.read_csv("../../data/processed/01.3/accidents_advance_clean.csv")

df_test = pd.read_csv("../../data/processed/01.2/test_features_for_predict.csv")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

Train shape: (5469092, 42)
Test shape: (1367273, 37)


### Feature Creation

In [4]:
import pandas as pd
import numpy as np
import gc

def advanced_feature_creation_v2(df):
    # 1. จัดการข้อมูลเชิงเวลา (Temporal Features)
    if 'Start_Time' in df.columns:
        # แปลงเป็น datetime เฉพาะเมื่อจำเป็น
        if not pd.api.types.is_datetime64_any_dtype(df['Start_Time']):
            df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
        
        # ดึง Hour มาใช้เพื่อลดการเรียก dt.hour ซ้ำๆ
        hours = df['Start_Time'].dt.hour
        
        # Is_Night: ช่วงเที่ยงคืนถึงตี 5
        df['is_night'] = ((hours >= 0) & (hours <= 5)).astype('int8')
        
        # Is_Rush_Hour: ช่วงเช้า 6-9 และ เย็น 16-19
        df['is_rush_hour'] = (((hours >= 6) & (hours <= 9)) | 
                              ((hours >= 16) & (hours <= 19))).astype('int8')
        
        # Is_Weekend: วันเสาร์-อาทิตย์
        df['is_weekend'] = (df['Start_Time'].dt.dayofweek >= 5).astype('int8')

        df.drop(columns=['Start_Time'], inplace=True)
    else:
        print("คำเตือน: ไม่พบคอลัมน์ 'Start_Time' ข้ามการสร้างฟีเจอร์เชิงเวลา")

    # กลุ่มสภาพอากาศวิกฤต (Weather Risk)
    if 'Visibility(mi)' in df.columns:
        df['is_extreme_low_vis'] = (df['Visibility(mi)'] < 2).astype('int8')
    
    if 'Precipitation(in)' in df.columns and 'Weather_Group' in df.columns:
        df['is_slippery'] = ((df['Precipitation(in)'] > 0) & 
                             (df['Weather_Group'].isin(['rain', 'snow_ice']))).astype('int8')
    
    if 'Temperature(F)' in df.columns and 'Wind_Speed(mph)' in df.columns:
        
        df['temp_wind_interaction'] = (df['Temperature(F)'] * df['Wind_Speed(mph)']).astype('float32')

    # กลุ่มวิศวกรรมถนน (Infrastructure)
    poi_list = ['Junction', 'Traffic_Signal', 'Crossing']
    valid_poi = [c for c in poi_list if c in df.columns]
    if valid_poi:
        df['road_complexity'] = df[valid_poi].sum(axis=1).astype('int8')
    
    if 'Stop' in df.columns and 'Traffic_Signal' in df.columns:
        # ตรวจสอบเผื่อกรณีคอลัมน์ถูกแปลงเป็น One-hot ไปแล้ว
        try:
            df['is_high_speed_zone'] = ((df['Stop'] == 0) & (df['Traffic_Signal'] == 0)).astype('int8')
        except: pass

    # กลุ่มความรุนแรงและตำแหน่ง
    if 'Severity' in df.columns and 'Distance(mi)' in df.columns:
        df['severity_dist_prod'] = (df['Severity'] * df['Distance(mi)']).astype('float32')
        
    if 'State' in df.columns:
        rural_states = ['SD', 'OR', 'WV', 'MT']
        df['is_slow_state'] = df['State'].isin(rural_states).astype('int8')

    return df

In [5]:
# ประมวลผล Train Data
print("กำลังสร้างฟีเจอร์สำหรับ Train Data...")
df_train = advanced_feature_creation_v2(df_train)
gc.collect() 
# ประมวลผล Test Data
print("กำลังสร้างฟีเจอร์สำหรับ Test Data...")
df_test = advanced_feature_creation_v2(df_test)
gc.collect()

print("สร้างฟีเจอร์ใหม่เสร็จสิ้นทั้งหมดแล้ว")

กำลังสร้างฟีเจอร์สำหรับ Train Data...
กำลังสร้างฟีเจอร์สำหรับ Test Data...
สร้างฟีเจอร์ใหม่เสร็จสิ้นทั้งหมดแล้ว


* is_night (เกิดเหตุตอนกลางคืน): จับช่วงเวลา 00:00 - 05:00 น. เพราะจากข้อมูล EDA มักพบว่าอุบัติเหตุช่วงเวลานี้ใช้เวลาเคลียร์นานที่สุด (อาจเพราะมืด หรือเจ้าหน้าที่กู้ภัยมีน้อย)

* is_rush_hour (ชั่วโมงเร่งด่วน): จับช่วง 06:00-09:00 และ 16:00-19:00 น. ช่วงนี้รถติดมากแต่อาจจะเคลียร์ได้เร็วเพราะเป็นแค่การชนท้ายเบาๆ โมเดลจะได้แยกแยะแพทเทิร์นนี้ได้

* is_weekend (วันหยุดสุดสัปดาห์): แยกวันเสาร์-อาทิตย์ ออกจากวันธรรมดา เพราะปริมาณรถบรรทุกหรือรูปแบบการเดินทางจะเปลี่ยนไป ซึ่งมีผลต่อระยะเวลาจัดการจราจร

* is_extreme_low_vis (ทัศนวิสัยแย่มาก): ระบุจุดที่มองเห็นได้น้อยกว่า 2 ไมล์ (เช่น หมอกลงจัด) ทำให้การทำงานของเจ้าหน้าที่ยากและอันตรายขึ้น

* is_slippery (ถนนลื่น): ระบุเหตุการณ์ที่มีฝนตก/หิมะตก ร่วมกับกลุ่มสภาพอากาศที่ทำให้ถนนลื่น การย้ายรถที่เกิดเหตุบนถนนลื่นย่อมใช้เวลานานกว่าปกติ

* temp_wind_interaction (ดัชนีความลำบากจากอากาศ): เอาอุณหภูมิมาคูณกับความเร็วลม ยิ่งอุณหภูมิต่ำ (หนาวจัด) แล้วลมแรงด้วย จะสะท้อนถึงอุปสรรคทางกายภาพที่ทำให้เจ้าหน้าที่กู้ภัยทำงานช้าลงครับ

* road_complexity (คะแนนความวุ่นวายของถนน): เอาค่าป้ายเตือนต่างๆ (ทางแยก + ไฟจราจร + ทางม้าลาย) มาบวกกัน ได้คะแนน 0-3 ยิ่งคะแนนสูงแปลว่าเป็นสี่แยกใหญ่ที่เคลียร์พื้นที่ยาก

* is_high_speed_zone (โซนใช้ความเร็ว/ไฮเวย์): หากจุดเกิดเหตุไม่มีทั้ง "ป้ายหยุด" และ "ไฟจราจร" สันนิษฐานได้ว่าเป็นถนนทางหลวงวิ่งยาวๆ ซึ่งอุบัติเหตุในโซนนี้มักจะรุนแรงและกินพื้นที่กว้างกว่าถนนในซอยครับ

* severity_dist_prod (ผลคูณความรุนแรงและระยะทาง): เอาความรุนแรง (1-4) มาคูณขยายกับระยะทางที่รถติด (ไมล์) เพื่อสร้าง "ดัชนีผลกระทบรวม" ยิ่งค่านี้สูง โมเดลจะยิ่งรู้ทันทีว่านี่คือ "งานช้าง" ที่ต้องใช้เวลาเคลียร์นานแน่ๆ

* is_slow_state (พื้นที่เคลียร์ช้า): ดึงข้อมูลรัฐที่มักจะใช้เวลาจัดการจราจรนานเป็นพิเศษ (เช่น SD, OR, WV, MT) มาปักธงไว้ เพื่อให้โมเดลให้น้ำหนักกับพื้นที่เหล่านี้เพิ่มขึ้นครับ

### Feature Selection

In [6]:
# รายชื่อคอลัมน์ที่ต้องการลบทิ้ง
cols_to_drop = ['Wind_Chill(F)', 'Start_Time', 'Start_Date', 'City'] 

# ลบออกจากทั้ง Train และ Test
df_train = df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns], errors='ignore')
df_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns], errors='ignore')

Boolean Encoding

In [7]:
# หาคอลัมน์ที่เป็น Boolean ทั้งหมด
bool_cols = df_train.select_dtypes(include=['bool']).columns

df_train[bool_cols] = df_train[bool_cols].astype(int)
df_test[bool_cols] = df_test[bool_cols].astype(int)

Target Encoding สำหรับชื่อรัฐ (State)

In [8]:
# คำนวณค่า Median Duration ของแต่ละรัฐจาก "Train Data" เท่านั้น!
state_target_median = df_train.groupby('State')['Duration(min)'].median().to_dict()

# นำ Dictionary ที่ได้ไป Map ใส่คอลัมน์ State ทั้งใน Train และ Test
df_train['State_Encoded'] = df_train['State'].map(state_target_median)
df_test['State_Encoded'] = df_test['State'].map(state_target_median)

# หากใน Test มีรัฐที่ไม่มีใน Train ให้เติมด้วยค่า Median รวมของ Train
global_median = df_train['Duration(min)'].median()
df_test['State_Encoded'] = df_test['State_Encoded'].fillna(global_median)

# ลบคอลัมน์ State แบบข้อความทิ้ง
df_train = df_train.drop(columns=['State'])
df_test = df_test.drop(columns=['State'])

One-Hot Encoding

In [9]:
cols_to_encode = ['Severity', 'Weather_Group', 'Sunrise_Sunset']

# ทำ One-Hot Encoding
df_train = pd.get_dummies(df_train, columns=[c for c in cols_to_encode if c in df_train.columns], drop_first=False)
df_test = pd.get_dummies(df_test, columns=[c for c in cols_to_encode if c in df_test.columns], drop_first=False)

# จัดการให้คอลัมน์ใน Train และ Test ตรงกัน (เผื่อ Test ขาดบาง Category)
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

# ลบคอลัมน์ Target ออกจาก Test เพราะคำสั่ง align ทำให้มันโผล่มาเป็น 0
if 'Duration(min)' in df_test.columns:
    df_test = df_test.drop(columns=['Duration(min)', 'Distance(mi)'], errors='ignore')

Feature Scaling

In [10]:
# แยกคอลัมน์เป้าหมายออกจาก Train ก่อน
y_duration = df_train['Duration(min)']
y_distance = df_train['Distance(mi)']
X_train = df_train.drop(columns=['Duration(min)', 'Distance(mi)'])

# คอลัมน์ที่ต้องการทำ Scaling (ตัวเลขต่อเนื่อง)
num_cols_to_scale = ['Start_Lat', 'Start_Lng', 'Temperature(F)', 'Humidity(%)', 
                     'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 
                     'Precipitation(in)', 'State_Encoded',
                     'temp_wind_interaction', 'road_complexity', 'severity_dist_prod']

# สร้าง Object StandardScaler
scaler = StandardScaler()

# "Fit" (เรียนรู้สเกล) และ "Transform" (แปลง) บน Train Data
X_train[num_cols_to_scale] = scaler.fit_transform(X_train[num_cols_to_scale])

# "Transform" (แปลงอย่างเดียว โดยใช้สเกลที่เรียนรู้จาก Train) บน Test Data
df_test[num_cols_to_scale] = scaler.transform(df_test[num_cols_to_scale])